In [5]:
# Cell 1
import sys, os, json, random, glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
import matplotlib.patches as patches




import json


import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("Python:", sys.version)
print("OK imports")

Python: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
OK imports


In [6]:
# Cell 2 — Set dataset paths
# Edit these to match your dataset structure.
# Examples:
#   dataset_root/
#     images/
#     annotations/
#       instances_train.json   (COCO)
#     labels/                  (YOLO)
#     Annotations/             (VOC)

# Cell 2
DATASET_ROOT = Path("/projets/Fbassignana/diffusers_try/flow_matching_trial/v18")  # <-- CHANGE ME

# Common folder names (we'll search these if they exist)
CANDIDATE_IMAGE_DIRS = ["images", "imgs", "JPEGImages", "Image", "Images"]
CANDIDATE_ANN_DIRS   = ["annotations", "annotation", "Annotations", "labels", "Label", "Labels"]

# If you know exact paths, you can set them explicitly:
IMAGE_DIR = Path("/projets/Fbassignana/diffusers_try/flow_matching_trial/v18/images")   # e.g., DATASET_ROOT / "images"
ANN_DIR   = IMAGE_DIR / "annotations"   # e.g., DATASET_ROOT / "annotations"

print("DATASET_ROOT:", DATASET_ROOT)

DATASET_ROOT: /projets/Fbassignana/diffusers_try/flow_matching_trial/v18


In [ ]:

class BBoxConditioningDataset(Dataset):
    """
    Loads ``.npy`` images together with their COCO-format bounding-box
    annotations and returns both the pixel tensor and a binary bbox mask.

    Returns
    -------
    dict
        ``pixel_values``               – (C, 256, 256) normalised to [-1, 1]
        ``conditioning_pixel_values``   – (1, 256, 256) binary bbox mask
    """

    def __init__(
        self,
        root_dir: str,
        annotations_path: str,
        conditioning_dropout: float = 0.0,
        transform = None
    ):
        self.transform = transform
        self.root_dir = root_dir
        self.conditioning_dropout = conditioning_dropout

        # Enumerate .npy files
        self.files = sorted(
            f for f in os.listdir(root_dir) if f.endswith(".npy")
        )
        if not self.files:
            raise RuntimeError(f"No .npy files found in {root_dir}")

        # Parse COCO annotations
        with open(annotations_path, "r") as fh:
            data = json.load(fh)

        id_to_fname: dict[str, str] = {}
        self.img_info: dict[str, dict] = {}
        for img in data["images"]:
            id_to_fname[img["id"]] = img["file_name"]
            self.img_info[img["file_name"]] = {
                "width": img["width"],
                "height": img["height"],
                "boxes": [],
            }

        for annot in data["annotations"]:
            fname = id_to_fname.get(annot["image_id"])
            if fname is not None and fname in self.img_info:
                self.img_info[fname]["boxes"].append(annot["bbox"])

        print(
            f"[BBoxDataset] {len(self.files)} files, "
            f"{len(self.img_info)} annotated images"
        )

    def __len__(self) -> int:
        return len(self.files)

    # ------------------------------------------------------------------ #

    @staticmethod
    def _make_bbox_mask(
        boxes: list, width: int, height: int
    ) -> torch.Tensor:
        """Rasterise ``[x, y, w, h]`` boxes into a binary (1, H, W) mask."""
        mask = torch.zeros(1, height, width)
        for bx, by, bw, bh in boxes:
            x0 = max(0, int(bx))
            y0 = max(0, int(by))
            x1 = min(width, int(bx + bw + 0.5))
            y1 = min(height, int(by + bh + 0.5))
            mask[0, y0:y1, x0:x1] = 1.0
        return mask

    # ------------------------------------------------------------------ #

    def __getitem__(self, idx: int) -> dict:
        fname = self.files[idx]

        # --- image -------------------------------------------------------
        x = np.load(os.path.join(self.root_dir, fname))
        if x.ndim == 2:
            x = x[None, ...]  # (1, H, W)
        x = torch.from_numpy(x).float()

        # --- bbox mask ---------------------------------------------------
        info = self.img_info.get(fname)
        if info is not None:
            w, h = info["width"], info["height"]
            boxes = info["boxes"]
        else:
            _, h, w = x.shape
            boxes = []

        mask = self._make_bbox_mask(boxes, w, h)

        # --- resize to 256×256 ------------------------------------------
        if self.transform:
            x = self.transform(x)

        mask = F.interpolate(
            mask.unsqueeze(0), size=(256, 256), mode="nearest"
        ).squeeze(0)

        # --- conditioning dropout ----------------------------------------
        if (
            self.conditioning_dropout > 0
            and torch.rand(()).item() < self.conditioning_dropout
        ):
            mask = torch.zeros_like(mask)

        return {
            "pixel_values": x,
            "conditioning_pixel_values": mask,
        }


In [9]:
dataset = BBoxConditioningDataset(
    root_dir=str(IMAGE_DIR),
    annotations_path=str(ANN_DIR / "annotations.json"),
    conditioning_dropout=0.1,
)
print("Dataset size:", len(dataset))
sample = dataset[0]
print("Sample keys:", sample.keys())
print("pixel_values shape:", sample["pixel_values"].shape)
print("conditioning_pixel_values shape:", sample["conditioning_pixel_values"].shape)


[BBoxDataset] 6725 files, 6725 annotated images
Dataset size: 6725
Sample keys: dict_keys(['pixel_values', 'conditioning_pixel_values'])
pixel_values shape: torch.Size([1, 256, 256])
conditioning_pixel_values shape: torch.Size([1, 256, 256])
